# Create Compagnia di San Paolo Awards (research grants only)

Creates Fondazione Compagnia di San Paolo (Turin) research grants from the
foundation's own annual **"Elenco Beneficiari"** PDFs, linked from
https://www.compagniadisanpaolo.it/it/documenti-istituzionali/trasparenza.
CSP publishes no grants database, API or export; these PDFs are its only
grant-level publication (beneficiary, seat, project title, EUR amount).

**Research filter:** CSP is a large general philanthropic foundation. Only
grants listed under the Planet-goal mission **"Valorizzare la ricerca"**
(Harnessing the value of research: research calls, doctoral and fellowship
funding, research equipment and infrastructure, research institutes and
university research agreements) are ingested. All culture, social, welfare,
innovation, international and environment missions are excluded, as are rows
whose beneficiary is CSP itself / "Progetti operativi e accantonamenti"
(internal operating costs and provisions, incl. the aggregate TRAPEZIO call
allocations) and any "accantonamento" row. The mission also contains some
university capacity-building and outreach items; they are kept because CSP
classifies them as research-mission grants.

**Years: 2023-2025 only (179 grants, EUR 49.2M).** 2019-2022 lists use other
layouts and a different sector taxonomy, and 2015-2018 publish only
per-beneficiary totals, so they are not ingested.

**No PI names** are published: `lead_investigator` carries only the
beneficiary organisation as `affiliation.name` (given/family NULL).
**Dates:** deliberation year only (`start_date` = Jan 1).

**Prerequisites:**
- Run `scripts/local/compagnia_san_paolo_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/compagnia_san_paolo/compagnia_san_paolo_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** CSP prints no grant number in the lists.
Citing works (`works?filter=grants.funder:F4320309746&select=grants`) quote
internal request/protocol numbers (`2018.0059`, `CSTO164394`, `ID ROL 9361`)
that are not published, so the key is synthetic:
`CSP-{year}-{10-hex hash of beneficiary|title|amount}`. Expect 0 collapse onto
existing citation stubs.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320309746`
- display_name / ror_id / doi: from `openalex.funders.funders` (Compagnia di San Paolo, ror 00d1b1c41, doi 10.13039/100007388)

**Priority:** `512` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.compagnia_san_paolo_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/compagnia_san_paolo/compagnia_san_paolo_projects.parquet`;

In [ ]:
%sql
SELECT year, COUNT(*) as n, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as eur
FROM openalex.awards.compagnia_san_paolo_raw GROUP BY year ORDER BY year;

In [ ]:
%sql
DESCRIBE openalex.awards.compagnia_san_paolo_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.compagnia_san_paolo_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320309746 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309746;

## Step 2: Create Compagnia di San Paolo Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.compagnia_san_paolo_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309746  -- Compagnia di San Paolo
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Obiettivo Pianeta - Missione ', g.mission) as funder_scheme,
    'compagnia_san_paolo_beneficiari' as provenance,
    TRY_TO_DATE(CONCAT(TRY_CAST(g.year AS INT), '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- Beneficiary organisation only (no person names are published).
    CASE WHEN NULLIF(TRIM(g.beneficiary), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.beneficiary),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>)
        as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>)
        as investigators,
    g.source_pdf as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.compagnia_san_paolo_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is the synthetic CSP-{year}-{hash} key.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^CSP-20[0-9]{2}-[0-9A-F]{10}(-[0-9]+)?$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^CSP-20[0-9]{2}-[0-9A-F]{10}(-[0-9]+)?$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.compagnia_san_paolo_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'compagnia_san_paolo_beneficiari' AND priority = 512;

-- Insert into openalex_awards_raw with priority.
-- Priority 512: direct-from-funder ingest of CSP's own annual beneficiary lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    512 as priority
FROM openalex.awards.compagnia_san_paolo_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_csp_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.compagnia_san_paolo_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year,
       lead_investigator.affiliation.name
FROM openalex.awards.compagnia_san_paolo_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.compagnia_san_paolo_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: beneficiary organisations and titles should be a long tail.
SELECT lead_investigator.affiliation.name AS beneficiary, COUNT(*) AS n
FROM openalex.awards.compagnia_san_paolo_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.compagnia_san_paolo_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- Coverage (expect 100% title/amount/start year/beneficiary, 0% person names).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.affiliation.name) as has_beneficiary
FROM openalex.awards.compagnia_san_paolo_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for CSP (expect 0: CSP publishes no grant numbers).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(n.id) as upgraded_by_csp_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.compagnia_san_paolo_awards n ON n.id = c.id
WHERE c.funder_id = 4320309746 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'compagnia_san_paolo_beneficiari'
GROUP BY provenance, priority;